# Super ii API and MCP quickstart

Read Super ii's production system-state contract and perform a public MCP handshake using only Python's standard library. This notebook sends no credentials and does not execute repository content.

In [ ]:
import json
from urllib.request import Request, urlopen

ORIGIN = "https://superii.site"

def get_json(path):
    request = Request(f"{ORIGIN}{path}", headers={"Accept": "application/json"})
    with urlopen(request, timeout=20) as response:
        return json.load(response)

## Read capability truth

The system-state resource separates implementation status from present availability. Do not infer that a feature is generally available merely because its code exists.

In [ ]:
state = get_json("/system-state.json")
print(f"Snapshot: {state['snapshot']}")
for capability in state['capabilities'][:6]:
    print(f"- {capability['capability']}: {capability['status']} / {capability['availability']}")

## Initialize the public MCP endpoint

Super ii's MCP surface is public, stateless, and read-only. The response may use JSON or Server-Sent Events, so the helper handles both representations.

In [ ]:
def mcp_request(method, params=None, request_id=1):
    payload = {"jsonrpc": "2.0", "id": request_id, "method": method}
    if params is not None:
        payload["params"] = params
    request = Request(
        f"{ORIGIN}/mcp",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Accept": "application/json, text/event-stream",
            "Content-Type": "application/json",
            "MCP-Protocol-Version": "2025-06-18",
        },
        method="POST",
    )
    with urlopen(request, timeout=20) as response:
        body = response.read().decode("utf-8")
        content_type = response.headers.get("content-type", "")
    if "text/event-stream" in content_type:
        events = [line[6:] for line in body.splitlines() if line.startswith("data: ")]
        if not events:
            raise RuntimeError("MCP response did not contain a data event")
        body = events[-1]
    return json.loads(body)

In [ ]:
handshake = mcp_request(
    "initialize",
    {
        "protocolVersion": "2025-06-18",
        "capabilities": {},
        "clientInfo": {"name": "super-ii-official-notebook", "version": "1.0"},
    },
)
print(json.dumps(handshake.get("result", handshake), indent=2)[:2000])

## Next steps

Use `/api/search?kind=model`, `/api/search?kind=dataset`, or `/api/search?kind=space` for bounded public discovery. Every reviewed repository publishes immutable checksums and machine-readable README, agents, manifest, API, and MCP representations. The catalogs may truthfully return zero items until the first reviewed upload is approved.